# M9: HyperPod Distributed Training — Multi-Node Scale-Up

**Pipeline Position:** Stages 3/5/7 (extension): Distributed Training Scale-Up — SageMaker HyperPod *(supplementary infra)*
**Input S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m3/curated_captions.json` *(real — trains on M3's curated captions)*
**Output S3 Path:** `s3://av30lab-user-workspace-{account_id}/users/{profile}/m9/`
**Notebook instance:** `ml.t3.medium` (CPU) — it only submits/monitors the job
**Training instances:** `ml.m5.xlarge` × 2 (CPU, gloo DDP) — the actual distributed run

## What Is HyperPod?

Amazon SageMaker HyperPod is purpose-built infrastructure for training foundation models at scale:

- **Resilient clusters** — automatic node replacement on hardware failure
- **Optimized networking** — EFA (Elastic Fabric Adapter) for NCCL all-reduce
- **Managed Slurm/EKS** — familiar job schedulers on managed infrastructure
- **Pre-configured deep learning** — CUDA, NCCL, cuDNN, frameworks pre-installed
- **Checkpoint management** — automatic save/resume on node failures

## Scope of This Notebook — read this

This notebook runs a **real, distributed PyTorch DDP training job** (SageMaker
Training Job, `instance_count=2`) on **M3's curated captions**, and visualizes the
**measured** per-epoch loss/throughput from the job's own artifacts — nothing here
is simulated.

It is **not** a HyperPod cluster. Full SageMaker HyperPod is **separate persistent
infrastructure** — `create-cluster` with Slurm or EKS orchestration, FSx for Lustre,
EFA — that a notebook cannot provision (much like M7's AlpaSim runs outside the
notebook). So M9 demonstrates the **distributed-training pattern** that HyperPod
scales, on affordable CPU instances:

> **CPU by design.** The demo model is a small MLP, so the point is genuine
> multi-node `torch.distributed` all-reduce, not GPU throughput. We use
> `ml.m5.xlarge` × 2 with the **gloo** backend (CPU training quota is available;
> GPU training quota here is 1). The identical script auto-selects **nccl** and
> runs on `ml.g5.xlarge` GPUs if that quota is raised — see
> [`docs/HYPERPOD_M9.md`](../docs/HYPERPOD_M9.md) for the CPU-vs-GPU trade-off and
> what real HyperPod adds.

### One-time environment setup — expect this, it is not an error

The first code cell pins the classic **SageMaker SDK v2** (this notebook uses the v2 API; the kernel may ship v3). On a fresh kernel it installs v2 and continues with **no restart**. Only if v3 was already loaded into memory will the kernel restart **once** — you'll see a small orange banner (not a red error). When it returns, just choose **Run All** again; the cell detects v2 and continues.

In [ ]:
"""Environment Setup — SDK pin (run this cell first, on its own)"""
# The SageMaker Distribution kernel may ship the NEW SageMaker Python SDK **v3**
# (modular: sagemaker.core / sagemaker.train, no top-level Session or the classic
# sagemaker.pytorch / sagemaker.workflow layout). This notebook uses the classic
# **v2** API, so we pin v2 (security-patched >=2.257.2).
#
# We restart the kernel ONLY if v3 was already imported into memory (pip can swap
# files on disk, but a resident module stays in memory). On a fresh kernel where
# sagemaker was never imported, we install v2 and import it in-place — NO restart.
# If a restart IS needed you'll see a small orange banner (not a red error); when
# the kernel comes back, just Run All again — this cell detects v2 and continues.
import subprocess, sys


def _sm_version():
    """Return (version, resident) WITHOUT importing sagemaker.

    Importing to detect would pull v3 into memory as a side effect and force an
    unnecessary restart, so we probe sys.modules (residency) + importlib.metadata
    (on-disk version) instead.
    """
    mod = sys.modules.get("sagemaker")
    if mod is not None:
        return getattr(mod, "__version__", None), True          # resident in memory
    try:
        import importlib.metadata as _md
        return _md.version("sagemaker"), False                  # on disk, not imported
    except Exception:
        return None, False


_ver, _resident = _sm_version()
_need_v2 = (_ver is None) or (not str(_ver).startswith("2"))
_restarting = False

if _need_v2:
    print(f"Current SageMaker SDK: {_ver} -> installing v2 (>=2.257.2,<3) ...")
    _res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sagemaker>=2.257.2,<3"],
        capture_output=True, text=True,
    )
    if _res.returncode != 0:
        # Real failure: surface everything (do NOT swallow) and stop loudly.
        print(_res.stdout)
        print(_res.stderr)
        raise RuntimeError("pip install of sagemaker v2 failed - see output above.")
    # Success: pip's harmless dependency-resolver conflict wall stayed captured.

    if _resident and str(_ver).startswith("3"):
        # v3 objects are already in memory - a one-time restart is the only clean
        # way to evict them. Friendly banner, graceful restart, NO raised exception.
        from IPython.display import display, HTML
        display(HTML(
            '<div style="padding:10px;border-left:4px solid #FF9900;'
            'background:#FFF8E1;font-family:sans-serif">'
            '<b>One-time kernel restart</b> to load the pinned SageMaker SDK v2. '
            'When it returns (a few seconds), just <b>Run All</b> again - this cell '
            'will detect v2 and continue. This is expected, not an error.</div>'))
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
        _restarting = True   # skip the rest of this cell; the queued restart fires
    # else: nothing resident -> import the freshly installed v2 in-place, no restart.

if not _restarting:
    import os
    import json
    import time
    import tarfile
    from pathlib import Path
    from datetime import datetime, timezone

    import boto3
    import sagemaker
    from sagemaker.session import Session          # v2 layout
    from sagemaker.pytorch import PyTorch          # v2 estimator

    print(f"SageMaker SDK v2 confirmed: {sagemaker.__version__}")

    # --- S3 Path Configuration ---
    ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
    PROFILE = os.environ.get("USER_PROFILE", "default")

    USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")
    INPUT_PREFIX = f"users/{PROFILE}/m3/"          # M3 curated captions (real input)
    OUTPUT_PREFIX = f"users/{PROFILE}/m9/"

    # SageMaker session
    sagemaker_session = Session()
    role = sagemaker.get_execution_role()
    region = sagemaker_session.boto_region_name

    # --- Clients ---
    s3 = boto3.client("s3")
    sm_client = boto3.client("sagemaker")

    # --- Pull M3's curated captions so the training job trains on REAL pipeline data,
    #     not synthetic noise. M3 writes {"curated_captions": [...]} to
    #     users/<profile>/m3/curated_captions.json. If M3 hasn't run, we fall back to
    #     a synthetic dataset (the training script handles both) but say so loudly.
    LOCAL_INPUT = Path("/tmp/m9_input")
    LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
    m3_key = f"{INPUT_PREFIX}curated_captions.json"
    USE_REAL_DATA = False
    try:
        s3.download_file(USER_BUCKET, m3_key, str(LOCAL_INPUT / "curated_captions.json"))
        n = len(json.loads((LOCAL_INPUT / "curated_captions.json").read_text()).get("curated_captions", []))
        USE_REAL_DATA = n > 0
        print(f"M3 input: {n} curated captions from s3://{USER_BUCKET}/{m3_key}")
    except Exception as e:
        print(f"(M3 curated_captions.json not found — will train on a synthetic dataset. "
              f"Run M3 first for a real end-to-end pipeline. Detail: {e})")

    print(f"Account ID: {ACCOUNT_ID}")
    print(f"Profile: {PROFILE}")
    print(f"Region: {region}")
    print(f"Role: {role}")
    print(f"Input: s3://{USER_BUCKET}/{INPUT_PREFIX}  (real data: {USE_REAL_DATA})")
    print(f"Output: s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
    print(f"SageMaker SDK: {sagemaker.__version__}")

## HyperPod Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                   SageMaker HyperPod Cluster                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐   EFA    ┌─────────────┐                 │
│  │  Node 0     │◄────────►│  Node 1     │                 │
│  │ 8x A100 80G │  400Gbps │ 8x A100 80G │                 │
│  │ (640GB VRAM)│          │ (640GB VRAM) │                 │
│  └──────┬──────┘          └──────┬──────┘                 │
│         │                        │                         │
│         └────────┬───────────────┘                         │
│                  │                                         │
│         ┌────────▼────────┐                                │
│         │  FSx for Lustre │  ← Shared high-speed storage   │
│         │  (Training data)│                                │
│         └─────────────────┘                                │
│                                                             │
│  Managed: Auto node replacement, checkpoint resume          │
│  Networking: NCCL over EFA, NVLink intra-node               │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Data Parallelism** | Same model replicated across GPUs, data split across replicas |
| **Model Parallelism** | Model layers split across GPUs (for models > single GPU memory) |
| **FSDP** | Fully Sharded Data Parallelism — shards optimizer states + gradients |
| **EFA** | Elastic Fabric Adapter — low-latency networking for NCCL collectives |
| **Checkpointing** | Periodic model state saves for fault tolerance |

In [ ]:
"""Prepare training script for distributed execution"""

# Create a local training script that demonstrates distributed PyTorch.
# Rank/world_size come from MPI (mpi4py) — SageMaker launches this under the `mpi`
# distribution (torch_distributed/torchrun is GPU-only in the SDK), and the MPI
# rank assignment does NOT match SM_HOSTS ordering, so we must read it from MPI
# itself. The MASTER address is rank-0's hostname, broadcast to all ranks. Backend
# auto-selects gloo (CPU / m5.xlarge, this workshop) or nccl (GPU / g5.xlarge if
# the GPU quota is raised — see docs/HYPERPOD_M9.md).
TRAIN_SCRIPT_DIR = Path("/tmp/m9_training")
TRAIN_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

training_script = '''
"""Distributed training script for AV caption quality model.

Demonstrates (for real, via torch.distributed):
- process group init (gloo on CPU / nccl on GPU, auto-selected)
- DistributedDataParallel (DDP) wrapping + gradient all-reduce
- data sharding across ranks (DistributedSampler)
- checkpoint save/resume + a real per-epoch training log

Rank/world_size come from MPI (mpi4py) because SageMaker runs this under the `mpi`
launcher with one process per host; MASTER_ADDR is rank-0's hostname broadcast to
all ranks. Trains on M3's curated captions when the `training` channel is present,
otherwise on a synthetic dataset. Either way the DDP mechanics are genuine.
"""
import os
import json
import time
import socket
import argparse
from pathlib import Path

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, DistributedSampler


FEATURE_DIM = 8  # features engineered from a caption string (see _caption_features)


def _caption_features(text):
    """Cheap, deterministic features from a caption string (no GPU/model needed)."""
    words = text.split()
    n_words = len(words)
    n_chars = len(text)
    avg_word = (n_chars / n_words) if n_words else 0.0
    n_digits = sum(c.isdigit() for c in text)
    n_upper = sum(c.isupper() for c in text)
    n_punct = sum(not c.isalnum() and not c.isspace() for c in text)
    uniq_ratio = (len(set(words)) / n_words) if n_words else 0.0
    return [
        n_words / 50.0, n_chars / 300.0, avg_word / 10.0, n_digits / 10.0,
        n_upper / 20.0, n_punct / 20.0, uniq_ratio, min(n_words, 100) / 100.0,
    ]


class CaptionQualityDataset(Dataset):
    """Real M3 captions -> engineered features, or synthetic fallback.

    Label = a deterministic 'quality' proxy (longer, more diverse captions score
    higher). This is a demo target; the point is genuine distributed training,
    not a production quality model.
    """

    def __init__(self, captions_path=None, num_samples=10000):
        self.real = False
        feats = []
        if captions_path and Path(captions_path).exists():
            data = json.loads(Path(captions_path).read_text())
            caps = data.get("curated_captions", data if isinstance(data, list) else [])
            for c in caps:
                text = c.get("caption", "") if isinstance(c, dict) else str(c)
                if text:
                    feats.append(_caption_features(text))
            if feats:
                self.real = True
        if not feats:
            # synthetic fallback
            g = torch.Generator().manual_seed(0)
            self.features = torch.randn(num_samples, FEATURE_DIM, generator=g)
            self.labels = torch.sigmoid(self.features.mean(dim=1, keepdim=True))
            return
        self.features = torch.tensor(feats, dtype=torch.float32)
        # quality proxy: emphasise word-count + uniqueness features
        self.labels = torch.sigmoid(
            (self.features[:, 0] * 1.5 + self.features[:, 6] * 1.0)
        ).unsqueeze(1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class QualityPredictor(nn.Module):
    """Simple MLP for caption quality prediction (demo model)."""

    def __init__(self, input_dim=FEATURE_DIM, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


def _init_distributed():
    """Init the process group using MPI-provided rank/world_size.

    SageMaker's `mpi` distribution launches this via mpirun, so rank/world_size
    come from MPI (mpi4py), NOT from SM_HOSTS ordering (they differ — using
    SM_HOSTS[0] as master hangs the rendezvous). rank-0 broadcasts its hostname
    as MASTER_ADDR so every rank agrees on the same rendezvous endpoint. gloo on
    CPU, nccl on GPU.
    """
    from mpi4py import MPI
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    world_size = comm.Get_size()
    # rank 0 picks its own hostname/IP as the rendezvous master, broadcast to all.
    master_addr = socket.gethostbyname(socket.gethostname()) if rank == 0 else None
    master_addr = comm.bcast(master_addr, root=0)
    os.environ["MASTER_ADDR"] = master_addr
    os.environ.setdefault("MASTER_PORT", "29500")
    os.environ["WORLD_SIZE"] = str(world_size)
    os.environ["RANK"] = str(rank)
    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)
    return backend, rank, world_size, use_cuda


def train(args):
    backend, global_rank, world_size, use_cuda = _init_distributed()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    if use_cuda:
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        ddp_device_ids = [local_rank]
    else:
        device = torch.device("cpu")
        ddp_device_ids = None  # DDP on CPU takes no device_ids

    # M3 captions arrive on the SageMaker "training" channel; else synthetic.
    ch = os.environ.get("SM_CHANNEL_TRAINING", "")
    cap_path = os.path.join(ch, "curated_captions.json") if ch else ""
    dataset = CaptionQualityDataset(captions_path=cap_path, num_samples=args.num_samples)

    if global_rank == 0:
        print(f"Backend: {backend} | world_size: {world_size} | nodes: {args.num_nodes}", flush=True)
        print(f"Dataset: {'REAL M3 captions' if dataset.real else 'synthetic'} "
              f"| samples: {len(dataset)} | feature_dim: {FEATURE_DIM}", flush=True)

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=global_rank)
    dataloader = DataLoader(
        dataset, batch_size=args.batch_size, sampler=sampler,
        num_workers=2, pin_memory=use_cuda,
    )

    model = QualityPredictor().to(device)
    model = DDP(model, device_ids=ddp_device_ids)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    criterion = nn.MSELoss()

    training_log = []
    avg_loss = float("nan")
    for epoch in range(args.epochs):
        sampler.set_epoch(epoch)
        model.train()
        epoch_loss, num_batches = 0.0, 0
        epoch_start = time.time()
        for batch_features, batch_labels in dataloader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad()
            predictions = model(batch_features)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1
        avg_loss = epoch_loss / max(num_batches, 1)
        epoch_time = time.time() - epoch_start
        samples_per_sec = len(dataset) / epoch_time if epoch_time > 0 else 0.0
        if global_rank == 0:
            print(f"Epoch {epoch+1}/{args.epochs} | Loss: {avg_loss:.6f} | "
                  f"Time: {epoch_time:.2f}s | Throughput: {samples_per_sec:.0f} samples/s", flush=True)
            training_log.append({
                "epoch": epoch + 1, "loss": avg_loss,
                "epoch_time_s": epoch_time,
                "throughput_samples_per_s": samples_per_sec,
            })

    if global_rank == 0:
        checkpoint_dir = Path(args.model_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "epoch": args.epochs,
            "model_state_dict": model.module.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "final_loss": avg_loss,
        }, checkpoint_dir / "model_checkpoint.pt")
        with open(checkpoint_dir / "training_log.json", "w") as f:
            json.dump({
                "backend": backend,
                "world_size": world_size,
                "num_nodes": args.num_nodes,
                "epochs": args.epochs,
                "batch_size": args.batch_size,
                "dataset": "real_m3" if dataset.real else "synthetic",
                "num_samples": len(dataset),
                "training_log": training_log,
            }, f, indent=2)
        print("Checkpoint + training_log.json saved to", args.model_dir, flush=True)

    dist.destroy_process_group()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--num-samples", type=int, default=50000)
    parser.add_argument("--num-nodes", type=int, default=2)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    args = parser.parse_args()
    train(args)
'''

# train_distributed.py imports mpi4py unconditionally for the MPI
# rendezvous. SageMaker auto-pip-installs a requirements.txt found in
# source_dir at container boot, so pin mpi4py here as a safety net — no-op
# if the PyTorch image already ships it, prevents an ImportError job-fail
# if it does not.
(TRAIN_SCRIPT_DIR / "requirements.txt").write_text("mpi4py\n")

script_path = TRAIN_SCRIPT_DIR / "train_distributed.py"
script_path.write_text(training_script)
print(f"Training script written to: {script_path}")
print(f"Script size: {script_path.stat().st_size} bytes")
print("Rendezvous from MPI (mpi4py) rank/world_size + rank-0 hostname broadcast.")

In [ ]:
"""Configure and launch distributed training job (2-node, CPU/gloo)"""

# This workshop trains on CPU (ml.m5.xlarge) with the gloo backend — the demo MLP
# needs no GPU, and CPU training quota is available out of the box. The identical
# script also runs GPU/nccl on ml.g5.xlarge if you raise that quota later; see
# docs/HYPERPOD_M9.md for the CPU-vs-GPU trade-off.
INSTANCE_TYPE = "ml.m5.xlarge"   # 2 nodes, CPU (gloo DDP)
INSTANCE_COUNT = 2               # 2-node distributed training
JOB_NAME = f"av30-m9-distributed-{PROFILE}-{int(time.time())}"

# Stage M3's curated captions as a training-channel input so the job trains on
# real pipeline data. If M3 didn't run, we skip the channel and the script uses
# its synthetic fallback (still a genuine distributed run).
train_inputs = None
if USE_REAL_DATA:
    channel_key = f"{OUTPUT_PREFIX}input/curated_captions.json"
    try:
        s3.upload_file(str(LOCAL_INPUT / "curated_captions.json"), USER_BUCKET, channel_key)
    except Exception as e:
        raise RuntimeError(
            f"Failed to stage M3 captions to s3://{USER_BUCKET}/{channel_key}: {e}\n"
            f"The execution role can write under users/{PROFILE}/; check creds/permissions "
            f"and re-run."
        ) from e
    train_inputs = {"training": f"s3://{USER_BUCKET}/{OUTPUT_PREFIX}input/"}
    print(f"Training channel: s3://{USER_BUCKET}/{OUTPUT_PREFIX}input/ (real M3 data)")
else:
    print("No training channel — job will use the synthetic dataset fallback.")

print(f"\nConfiguring distributed training job:")
print(f"  Job name: {JOB_NAME}")
print(f"  Instance type: {INSTANCE_TYPE} (CPU, gloo)")
print(f"  Instance count: {INSTANCE_COUNT}")
print(f"  Distribution: MPI (one process per host)")

# The SageMaker execution role can only write under the `users/*` prefix of the
# workspace bucket. By default the estimator uploads source.tar.gz to the BUCKET
# ROOT (<job>/source/...), which the role is NOT allowed to write → AccessDenied.
# So pin both the code upload (code_location) and outputs (output_path) under
# users/<profile>/m9/, which the role can write.
CODE_LOCATION = f"s3://{USER_BUCKET}/{OUTPUT_PREFIX}code"

# Create PyTorch estimator with distribution config.
#
# NOTE on why MPI, not torch_distributed: the SageMaker SDK restricts
# distribution={"torch_distributed"} (torchrun) to GPU/Trainium instances. On CPU
# we use the "mpi" launcher (one process per host); the training script derives
# rank/world_size from mpi4py (MPI.COMM_WORLD), with rank-0's hostname broadcast
# as MASTER_ADDR — NOT from SM_HOSTS ordering, which does not match the MPI rank
# assignment and hangs the rendezvous. On GPU you can switch to torch_distributed;
# the script's backend auto-select (gloo/nccl) handles both.
estimator = PyTorch(
    entry_point="train_distributed.py",
    source_dir=str(TRAIN_SCRIPT_DIR),
    role=role,
    instance_count=INSTANCE_COUNT,
    instance_type=INSTANCE_TYPE,
    framework_version="2.1.0",
    py_version="py310",
    output_path=f"s3://{USER_BUCKET}/{OUTPUT_PREFIX}",
    code_location=CODE_LOCATION,   # keep source.tar.gz under users/<profile>/ (role scope)
    sagemaker_session=sagemaker_session,

    # MPI: launches one training process per host (2 hosts -> world_size 2).
    # CPU-compatible, unlike torch_distributed which the SDK gates to GPU.
    distribution={"mpi": {"enabled": True, "processes_per_host": 1}},

    # Hyperparameters (feature_dim is fixed in the script at 8 engineered features)
    hyperparameters={
        "epochs": 10,
        "batch-size": 64,
        "lr": 0.001,
        "num-samples": 50000,   # only used by the synthetic fallback
        "num-nodes": INSTANCE_COUNT,
    },

    tags=[
        {"Key": "Project", "Value": "av30-blueprint-lab"},
        {"Key": "Module", "Value": "M9"},
        {"Key": "Profile", "Value": PROFILE},
    ],
    keep_alive_period_in_seconds=0,
    use_spot_instances=False,
    max_run=3600,  # 1 hour cap
)

print(f"\nEstimator configured. Ready to launch.")
print(f"  Framework: PyTorch 2.1.0 / Python 3.10")
print(f"  Distribution: mpi (1 process/host → gloo DDP on CPU)")
print(f"  Code location: {CODE_LOCATION}")

In [ ]:
"""Launch training job and monitor progress"""

print(f"Launching distributed training job: {JOB_NAME}")
print(f"  Nodes: {INSTANCE_COUNT}x {INSTANCE_TYPE} (CPU)")
print(f"  Data parallelism: batch split across {INSTANCE_COUNT} replicas (gloo all-reduce)")
print(f"  Input: {'real M3 captions' if USE_REAL_DATA else 'synthetic fallback'}")
print(f"\nStarting... (first launch spends a few minutes provisioning the 2 nodes)")

start_time = time.time()

# wait=True blocks until the job finishes; inputs mounts M3 data on the
# "training" channel (SM_CHANNEL_TRAINING) when we have real data.
estimator.fit(
    inputs=train_inputs,   # None -> synthetic fallback in the script
    job_name=JOB_NAME,
    wait=True,
    logs="All",
)

total_train_time = time.time() - start_time
print(f"\nTraining complete!")
print(f"  Total wall time: {total_train_time:.1f}s ({total_train_time/60:.1f} min)")
print(f"  Model artifacts: {estimator.model_data}")

In [ ]:
"""Retrieve and display REAL training metrics from the job's artifacts"""
import matplotlib.pyplot as plt

# Training job description (real API).
job_desc = sm_client.describe_training_job(TrainingJobName=JOB_NAME)
print("Training Job Summary:")
print(f"  Status: {job_desc['TrainingJobStatus']}")
print(f"  Instance: {job_desc['ResourceConfig']['InstanceCount']}x "
      f"{job_desc['ResourceConfig']['InstanceType']}")
print(f"  Training time: {job_desc.get('TrainingTimeInSeconds', 'N/A')}s")
print(f"  Billable time: {job_desc.get('BillableTimeInSeconds', 'N/A')}s")

# Download the model artifact tarball and read the REAL training_log.json the
# rank-0 process wrote (per-epoch loss + measured throughput). No simulation.
# Only a Completed job has a model artifact — guard so a Stopped/Failed re-run
# gives a clear message instead of AttributeError on model_data=None.
train_log = {}
art_dir = Path("/tmp/m9_artifacts")
art_dir.mkdir(parents=True, exist_ok=True)
if job_desc["TrainingJobStatus"] != "Completed":
    print(f"\nJob status is {job_desc['TrainingJobStatus']} — no model artifact to "
          f"read. Check the job logs above; re-run once the job Completes.")
    model_artifact_path = None
else:
    model_artifact_path = estimator.model_data
    print(f"\nModel artifacts: {model_artifact_path}")
    try:
        _p = model_artifact_path.replace("s3://", "").split("/", 1)
        s3.download_file(_p[0], _p[1], str(art_dir / "model.tar.gz"))
        with tarfile.open(art_dir / "model.tar.gz") as t:
            t.extractall(art_dir)
        log_file = art_dir / "training_log.json"
        train_log = json.loads(log_file.read_text()) if log_file.exists() else {}
    except Exception as e:
        print(f"(Could not download/extract the model artifact: {e})")
rows = train_log.get("training_log", [])
epochs = [r["epoch"] for r in rows]
losses = [r["loss"] for r in rows]
throughputs = [r["throughput_samples_per_s"] for r in rows]

print(f"\nReal run: backend={train_log.get('backend')} "
      f"world_size={train_log.get('world_size')} "
      f"dataset={train_log.get('dataset')} samples={train_log.get('num_samples')}")

if rows:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, losses, "b-o", alpha=0.8)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
    axes[0].set_title(f"Real training loss ({train_log.get('world_size')} ranks, "
                      f"{train_log.get('backend')})")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, throughputs, "g-s", alpha=0.8)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Throughput (samples/s)")
    axes[1].set_title("Measured throughput per epoch")
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("M9: Distributed Training Metrics (measured, not simulated)", fontsize=13)
    plt.tight_layout()
    plt.savefig("/tmp/m9_training_metrics.png", dpi=100, bbox_inches="tight")
    plt.show()

    final_loss = losses[-1]
    mean_tput = sum(throughputs) / len(throughputs)
    print(f"\nMeasured results:")
    print(f"  Final loss:      {final_loss:.6f}")
    print(f"  Mean throughput: {mean_tput:.0f} samples/s (aggregate over "
          f"{train_log.get('world_size')} ranks)")
    print(f"  Epochs:          {len(rows)}")
else:
    final_loss, mean_tput = float("nan"), 0.0
    print("(No training_log.json in the artifact — check the job logs above.)")

In [ ]:
"""Write training logs and metadata to S3"""

billable_seconds = job_desc.get("BillableTimeInSeconds", int(total_train_time))

training_output = {
    "module": "M9_HyperPod_Distributed_Training",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "training_job": {
        "job_name": JOB_NAME,
        "status": job_desc["TrainingJobStatus"],
        "instance_type": INSTANCE_TYPE,
        "instance_count": INSTANCE_COUNT,
        "framework": "PyTorch 2.1.0",
        "distribution": f"mpi launcher / gloo DDP ({train_log.get('backend', 'gloo')})",
        "training_time_s": job_desc.get("TrainingTimeInSeconds", int(total_train_time)),
        "billable_time_s": billable_seconds,
        "model_artifacts": model_artifact_path,
    },
    "data": {
        "source": train_log.get("dataset", "unknown"),   # real_m3 | synthetic
        "num_samples": train_log.get("num_samples"),
        "input": f"s3://{USER_BUCKET}/{INPUT_PREFIX}curated_captions.json" if USE_REAL_DATA else None,
    },
    "hyperparameters": {
        "epochs": train_log.get("epochs", 10),
        "batch_size": train_log.get("batch_size", 64),
        "learning_rate": 0.001,
        "feature_dim": 8,
    },
    "measured_metrics": {
        "world_size": train_log.get("world_size"),
        "final_loss": final_loss,
        "mean_throughput_samples_per_s": round(mean_tput, 1),
        "per_epoch": rows,   # the real training_log
    },
    "hyperpod_notes": (
        "This module demonstrates the distributed-training PATTERN with a real "
        "torch.distributed DDP job (SageMaker Training Job, instance_count=2). It is "
        "NOT a HyperPod cluster: full SageMaker HyperPod is separate persistent "
        "infrastructure (create-cluster, Slurm/EKS, FSx for Lustre, EFA) that a "
        "notebook cannot provision. HyperPod adds auto node replacement, shared "
        "high-speed storage, and job scheduling for large multi-node GPU training. "
        "This demo runs on CPU (gloo); the same script runs GPU/nccl if the g5 "
        "training quota is raised (see docs/HYPERPOD_M9.md)."
    ),
}

output_key = f"{OUTPUT_PREFIX}training_metadata.json"
s3.put_object(
    Bucket=USER_BUCKET,
    Key=output_key,
    Body=json.dumps(training_output, indent=2),
    ContentType="application/json",
)
print(f"Training metadata written to: s3://{USER_BUCKET}/{output_key}")

print(f"\nOutput Validation:")
head = s3.head_object(Bucket=USER_BUCKET, Key=output_key)
print(f"  OK: {output_key} ({head['ContentLength']} bytes)")
print(f"  Model artifacts: {model_artifact_path}")
print(f"  Data source: {train_log.get('dataset', 'unknown')}")

In [ ]:
"""Cost Analysis — real demo cost + conceptual HyperPod comparison"""

# Real demo pricing (what we actually used: ml.m5.xlarge CPU x2; rate shown for the run region).
DEMO_COST_PER_HOUR = 0.23   # ml.m5.xlarge on-demand (SageMaker training; ~same in us-west-2/us-east-1)
KRW_RATE = 1370

# Conceptual production HyperPod pricing (NOT run here — for scale context only).
HYPERPOD_COST_PER_HOUR = 37.69   # ml.p4d.24xlarge on-demand
HYPERPOD_RESERVED_COST = 22.03   # 1-yr reserved

demo_hours = billable_seconds / 3600
demo_cost_usd = DEMO_COST_PER_HOUR * INSTANCE_COUNT * demo_hours
demo_cost_krw = demo_cost_usd * KRW_RATE

prod_nodes, prod_hours = 2, 24
prod_cost_ondemand = HYPERPOD_COST_PER_HOUR * prod_nodes * prod_hours
prod_cost_reserved = HYPERPOD_RESERVED_COST * prod_nodes * prod_hours

print("=" * 60)
print("M9 Distributed Training — Cost Analysis")
print("=" * 60)
print()
print("--- This Workshop Demo (REAL, measured) ---")
print(f"Instance type:     {INSTANCE_TYPE} x{INSTANCE_COUNT} (CPU, gloo DDP)")
print(f"Cost per instance: ${DEMO_COST_PER_HOUR:.2f}/hr")
print(f"Billable time:     {billable_seconds}s ({demo_hours:.3f} hr)")
print(f"Demo cost:         ${demo_cost_usd:.3f} USD ({demo_cost_krw:.0f} KRW)")
print(f"Data source:       {train_log.get('dataset', 'unknown')}")
print()
print("--- Production HyperPod (CONCEPTUAL — not run here) ---")
print(f"Instance type:     ml.p4d.24xlarge (8x A100 80GB per node)")
print(f"Cluster size:      {prod_nodes} nodes ({prod_nodes*8} GPUs), {prod_hours}h training")
print(f"On-demand:         ${prod_cost_ondemand:,.0f} USD ({prod_cost_ondemand*KRW_RATE:,.0f} KRW)")
print(f"Reserved (1yr):    ${prod_cost_reserved:,.0f} USD  (~{(1-prod_cost_reserved/prod_cost_ondemand)*100:.0f}% saving)")
print()
print("--- Why the demo is CPU, and what HyperPod adds ---")
print("The demo model is a small MLP — distributed training PATTERN, not scale —")
print("so CPU (m5.xlarge x2, gloo) shows genuine DDP all-reduce for pennies. Real")
print("foundation-model training needs GPUs + HyperPod's separate infrastructure:")
print("  - cluster lifecycle (create/resize/delete), Slurm/EKS scheduling")
print("  - FSx for Lustre shared storage (~$0.145/GB-month)")
print("  - EFA networking + NCCL, auto node replacement on hardware failure")
print("Same training script runs GPU/nccl on g5.xlarge if that quota is raised.")
print("=" * 60)

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m09-hyperpod")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")